In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "async-geotiff>=0.4",
#     "lazycogs",
#     "lonboard>=0.15.0",
#     "numpy>=2",
#     "obstore>=0.9.2",
#     "pillow>=12.1.1",
#     "requests",
#     "rustac",
#     "sidecar>=0.8.1",
# ]
# ///

# CDL across many tiles, reprojected, in Lonboard — via `lazycogs`

The companion notebook (`raster-cog-cdl-pc.ipynb`) renders one Planetary Computer CDL tile interactively in its native Albers CRS, with deck.gl-raster reprojecting in the browser.

This one takes the other path: use [`lazycogs`] to mosaic _many_ MPC CDL tiles into a single reprojected xarray DataArray (Web Mercator), then drop it onto a map as a `BitmapLayer`. Same stack underneath — [`rustac`] for STAC, [`async-geotiff`] for COG I/O, [`obstore`] for cloud storage — but the lazy-xarray path means we can cover anything from a county to CONUS without giving up the no-server, no-GDAL story.

[`lazycogs`]: https://github.com/developmentseed/lazycogs
[`rustac`]: https://github.com/stac-utils/rustac-py
[`async-geotiff`]: https://developmentseed.org/async-geotiff/
[`obstore`]: https://developmentseed.org/obstore/

## Run

```
uvx juv run cdl-lazycogs-mosaic.ipynb
```

Optional: set `PC_SDK_SUBSCRIPTION_KEY` for higher MPC rate limits.

In [2]:
import base64
import io
from urllib.parse import urlparse

import lazycogs
import numpy as np
import requests
import rustac
from obstore.auth.planetary_computer import PlanetaryComputerCredentialProvider
from obstore.store import AzureStore
from PIL import Image
from pyproj import Transformer
from sidecar import Sidecar

from lonboard import Map
from lonboard.layer import BitmapLayer

ModuleNotFoundError: No module named 'lazycogs'

## Pick a region and a target CRS

**The reprojection chain:** CDL COGs are in EPSG:5070 (Albers Equal Area Conic). STAC search needs lat/lon (EPSG:4326). `BitmapLayer` accepts lat/lon `bounds` but deck.gl renders in Web Mercator (EPSG:3857). So our choice is what CRS to render the *pixels* in: render in 3857 and deck.gl's lat/lon→Mercator conversion of the bounds lines up with our pixels (no visible distortion). Render in 4326 and deck.gl stretches the image vertically — barely noticeable below ~50°N, ugly toward the poles.

For Iowa (~42°N), 4326 is honestly fine. Below we default to 4326 for simplicity; flip `DST_CRS` to `"EPSG:3857"` if you go further north or want pixel-accurate Mercator.

In [ ]:
DST_CRS = "EPSG:4326"

# CONUS — full lower 48.
BBOX_4326 = (-125.0, 24.0, -66.5, 49.5)

# Pixel size in target CRS units. 4326 → degrees. 0.02° ≈ 2 km at mid-latitudes.
# CONUS at 0.02° ≈ 2925 × 1275 px ≈ 3.7 MP — comfortable for a single BitmapLayer.
# Drop to 0.005° for ~500 m if you want detail, ~60 MP, expect a wait.
RESOLUTION = 0.02 if DST_CRS == "EPSG:4326" else 2000.0

if DST_CRS == "EPSG:3857":
    to_3857 = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
    DST_BBOX = (*to_3857.transform(BBOX_4326[0], BBOX_4326[1]),
                *to_3857.transform(BBOX_4326[2], BBOX_4326[3]))
else:
    DST_BBOX = BBOX_4326

DST_BBOX

## Cache STAC items to a local geoparquet via `rustac`

`lazycogs.open()` reads from a geoparquet file (DuckDB-backed). `rustac.search_to` writes one in one call.

In [ ]:
PARQUET = "cdl_items.parquet"

await rustac.search_to(
    PARQUET,
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    collections=["usda-cdl"],
    bbox=list(BBOX_4326),
    datetime="2021",
    filter={"op": "=", "args": [{"property": "usda_cdl:type"}, "cropland"]},
)
PARQUET

'cdl_items.parquet'

In [ ]:
# Sanity check: how many items did the search land?
client = rustac.DuckdbClient()
items = client.search(PARQUET, bbox=list(BBOX_4326))
print(f"items in parquet: {len(items)}")
if items:
    a = items[0]["assets"]
    print(f"first item id: {items[0]['id']}")
    print(f"assets: {list(a)}")
    print(f"first cropland href: {a['cropland']['href']}")

items in parquet: 35
first item id: cropland_2021_73905_2362575_90000
assets: ['cropland', 'confidence', 'tilejson', 'rendered_preview']
first cropland href: https://landcoverdata.blob.core.windows.net/usda-cdl/tiles/73905/2021_30m_cdls_73905_2362575_90000.tif


## Configure the obstore + PC credential provider

All CDL COGs live under one Azure container, so one `AzureStore` + one credential provider covers every item. `path_from_href` tells lazycogs how to map a STAC asset href to a path _inside_ that store.

In [ ]:
store = AzureStore(
    credential_provider=PlanetaryComputerCredentialProvider(
        url="https://landcoverdata.blob.core.windows.net/usda-cdl/",
    ),
)


def path_from_href(href: str) -> str:
    # https://landcoverdata.blob.core.windows.net/usda-cdl/tiles/.../foo.tif
    # ->                                                       tiles/.../foo.tif
    return urlparse(href).path.split("/", 2)[2]

## Open the mosaic as a lazy xarray DataArray

lazycogs builds the output grid in our target CRS + resolution, identifies which COGs intersect each chunk, fetches only the byte ranges needed, and reprojects with `pyproj` + numpy. Nothing is read until we `.compute()` (or `.values`).

In [ ]:
da = lazycogs.open(
    PARQUET,
    bbox=DST_BBOX,
    crs=DST_CRS,
    resolution=RESOLUTION,
    bands=["cropland"],
    dtype="uint8",
    nodata=0,
    store=store,
    path_from_href=path_from_href,
    max_concurrent_reads=64,  # CONUS pulls thousands of tiles; raise to amortize TLS handshakes
)
da

## CDL colormap from Google Earth Engine's STAC catalog

GEE publishes the full CDL class table (value, crop name, hex color) as part of its STAC JSON. Pulling from there is cleaner than scraping the LUT off a COG and gives us crop names for free — we'll use them for the legend and as the basis for crop-type filtering.

In [ ]:
GEE_URL = "https://storage.googleapis.com/earthengine-stac/catalog/USDA/USDA_NASS_CDL.json"
gee_classes = requests.get(GEE_URL, timeout=30).json()["summaries"]["eo:bands"][0]["gee:classes"]

# (256, 4) RGBA lookup table. Unmapped indices stay (0,0,0,0).
cmap_array = np.zeros((256, 4), dtype=np.uint8)
crop_names: dict[int, str] = {}
for c in gee_classes:
    v = int(c["value"])
    h = c["color"].lstrip("#")
    cmap_array[v] = (int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16), 255)
    crop_names[v] = c["description"]

# Override: GEE maps class 0 ("Background") to opaque black. Make it transparent
# so we see the basemap, not a black hole, in non-cropland areas.
cmap_array[0] = (0, 0, 0, 0)

print(f"{len(crop_names)} CDL classes, e.g.", {k: crop_names[k] for k in (1, 5, 24, 176) if k in crop_names})

## Materialize, colorize, encode

`.values` triggers all the byte-range fetches and reprojection. For ~3000×4000 px at 500 m resolution over Iowa, this is small — for CONUS, bump `resolution` to 1000+ or use `chunks=...` to get a dask-backed array and `.persist()`.

In [ ]:
arr = da.isel(band=0, time=0).values  # (y, x) uint8

# Diagnostics — if everything is 0, the search/store/path wiring is wrong, not the colormap.
nonzero = int((arr != 0).sum())
print(f"shape={arr.shape}  nonzero={nonzero} ({100 * nonzero / arr.size:.1f}%)  unique={np.unique(arr)[:20]}")

# cmap_array is (256, 4) RGBA — unmapped classes (incl. 0) have alpha=0.
rgba = cmap_array[arr]

buf = io.BytesIO()
Image.fromarray(rgba, mode="RGBA").save(buf, format="PNG")
data_url = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
len(buf.getvalue()), arr.shape

shape=(640, 1320)  nonzero=0 (0.0%)  unique=[0]


(4806, (640, 1320))

## Display

`BitmapLayer` takes `bounds` as `[west, south, east, north]` in lon/lat — deck.gl handles the Mercator stretch. Because we rendered the image _in_ Web Mercator, the stretch lines up correctly.

In [ ]:
west, south, east, north = BBOX_4326
layer = BitmapLayer(image=data_url, bounds=[west, south, east, north])

sidecar = Sidecar(anchor="split-right")
m = Map(layer, height=800)
with sidecar:
    display(m)

## Scaling notes

- **County** (~50 km): `RESOLUTION = 100.0` — full 30 m gets you ~25 MP, fine in memory.
- **State**: `RESOLUTION = 500.0` — what's set here, ~12 MP over Iowa.
- **CONUS**: `RESOLUTION = 2000.0`, and switch to dask: `da = lazycogs.open(..., chunks={"y": 2048, "x": 2048})` then `arr = da.isel(band=0, time=0).compute().values`. Or skip the single-`BitmapLayer` path entirely and tile it.
- **Multi-year change detection**: drop the `datetime="2021"` filter; lazycogs returns a `(band, time, y, x)` cube you can diff.